# 04 - Ablation Study: Feature Reduction
## CSE-CIC-IDS2018 — Top-20 vs Top-15 vs Top-10

**Tujuan:** Bandingkan performa 3 model (XGBoost, RF, SVM) dengan jumlah fitur berbeda.

**Skenario:**
- All Features (baseline dari Notebook 03)
- Top-20 Features
- Top-15 Features
- Top-10 Features

**Dataset:** file_100 (largest) — split 80/20

**Input:** `experiment_results_03.pkl`, `cleaned_100.pkl`

In [ ]:
# Install dependencies (run once per session)
import sys
!{sys.executable} -m pip install scikit-learn xgboost matplotlib seaborn psutil -q
print('✓ Dependencies installed')

In [ ]:
import pandas as pd
import numpy as np
import pickle, os, gc, time, warnings
import psutil
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 10, 'figure.dpi': 120})

DATA_DIR = '../data/'
MODEL_DIR = '../models/'
RANDOM_SEED = 42
TEST_SIZE = 0.20

print('Loading experiment results from Notebook 03...')
with open(os.path.join(DATA_DIR, 'experiment_results_03.pkl'), 'rb') as f:
    prev_results = pickle.load(f)

top20_features = prev_results['top20_features']
top15_features = prev_results['top15_features']
top10_features = prev_results['top10_features']
all_feature_names = prev_results['feature_names']
label_mapping = prev_results['label_mapping']

print(f'All features: {len(all_feature_names)}')
print(f'Top-20: {top20_features}')
print(f'Top-15: {top15_features}')
print(f'Top-10: {top10_features}')

## 1. Setup Models & Load Data

In [ ]:
# Load cleaned_100 (dataset terbesar)
with open(os.path.join(DATA_DIR, 'cleaned_100.pkl'), 'rb') as f:
    data = pickle.load(f)

X_all = data['X']
y_all = data['y']

print(f'Dataset: {X_all.shape[0]:,} samples, {X_all.shape[1]} features')
print(f'Classes: {len(label_mapping)}')

# Get feature indices for each subset
def get_feature_indices(feature_subset, all_features):
    return [all_features.index(f) for f in feature_subset if f in all_features]

idx_top20 = get_feature_indices(top20_features, all_feature_names)
idx_top15 = get_feature_indices(top15_features, all_feature_names)
idx_top10 = get_feature_indices(top10_features, all_feature_names)

print(f'Index Top-20: {len(idx_top20)} features')
print(f'Index Top-15: {len(idx_top15)} features')
print(f'Index Top-10: {len(idx_top10)} features')

In [ ]:
# Model factories
n_classes = len(np.unique(y_all))

MODELS = {
    'XGBoost': lambda: XGBClassifier(
        n_estimators=200, max_depth=8, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        objective='multi:softprob', num_class=n_classes,
        eval_metric='mlogloss', random_state=RANDOM_SEED,
        n_jobs=-1, tree_method='hist'),
    'Random Forest': lambda: RandomForestClassifier(
        n_estimators=200, max_depth=20, min_samples_split=5,
        random_state=RANDOM_SEED, n_jobs=-1),
    'SVM': lambda: SVC(
        kernel='rbf', C=10, gamma='scale',
        decision_function_shape='ovr', probability=True,
        random_state=RANDOM_SEED)
}

# Feature scenarios
FEATURE_SCENARIOS = {
    f'All ({len(all_feature_names)})': None,  # None = use all
    'Top-20': idx_top20,
    'Top-15': idx_top15,
    'Top-10': idx_top10
}

print(f'Models: {list(MODELS.keys())}')
print(f'Feature scenarios: {list(FEATURE_SCENARIOS.keys())}')
print(f'Total experiments: {len(MODELS) * len(FEATURE_SCENARIOS)}')

## 2. Run Ablation Experiments

In [ ]:
def run_experiment(X, y, model_factory, model_name, scenario_name):
    """Train and evaluate a single experiment."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y)
    
    model = model_factory()
    
    # Training
    mem_before = psutil.Process(os.getpid()).memory_info().rss / (1024*1024)
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start
    mem_after = psutil.Process(os.getpid()).memory_info().rss / (1024*1024)
    
    # Inference
    start = time.time()
    y_pred = model.predict(X_test)
    inf_time = time.time() - start
    inf_per_10k = (inf_time / len(X_test)) * 10000
    
    # Probabilities
    y_prob = model.predict_proba(X_test) if hasattr(model, 'predict_proba') else None
    
    # Metrics
    acc = accuracy_score(y_test, y_pred) * 100
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0) * 100
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0) * 100
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0) * 100
    try:
        roc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted') * 100 if y_prob is not None else 0.0
    except:
        roc = 0.0
    
    # Model size
    tag = f"{model_name.lower().replace(' ','_')}_{scenario_name.lower().replace(' ','_').replace('(','').replace(')','')}"
    mpath = os.path.join(MODEL_DIR, f'{tag}.pkl')
    with open(mpath, 'wb') as f:
        pickle.dump(model, f)
    model_size = os.path.getsize(mpath) / (1024*1024)
    
    return {
        'model_name': model_name,
        'scenario': scenario_name,
        'n_features': X.shape[1],
        'accuracy': acc, 'precision': prec, 'recall': rec,
        'f1_score': f1, 'roc_auc': roc,
        'train_time': train_time, 'inference_10k': inf_per_10k,
        'model_size_mb': model_size, 'ram_mb': max(0, mem_after - mem_before),
        'y_test': y_test, 'y_pred': y_pred
    }

In [ ]:
print('='*70)
print(f'{"ABLATION STUDY: Feature Reduction":^70}')
print('='*70)

ablation_results = []

for scenario_name, feat_idx in FEATURE_SCENARIOS.items():
    X_scenario = X_all[:, feat_idx] if feat_idx is not None else X_all
    print(f'\n--- {scenario_name} ({X_scenario.shape[1]} features) ---')
    
    for model_name, model_factory in MODELS.items():
        # SVM too slow for large dataset with many features
        if model_name == 'SVM' and X_scenario.shape[0] > 50000:
            print(f'  {model_name:15s} | SKIPPED (too slow for {X_scenario.shape[0]:,} samples)')
            continue
        
        result = run_experiment(X_scenario, y_all, model_factory, model_name, scenario_name)
        ablation_results.append(result)
        print(f'  {model_name:15s} | F1={result["f1_score"]:.2f}% | Acc={result["accuracy"]:.2f}% | Time={result["train_time"]:.2f}s')

print(f'\nTotal experiments: {len(ablation_results)}')

## 3. Table Perbandingan Performa

In [ ]:
# TABLE 1: Performance
rows = []
for r in ablation_results:
    rows.append({
        'Model': r['model_name'],
        'Features': r['scenario'],
        'N_feat': r['n_features'],
        'Accuracy (%)': f"{r['accuracy']:.2f}",
        'Precision (%)': f"{r['precision']:.2f}",
        'Recall (%)': f"{r['recall']:.2f}",
        'F1-Score (%)': f"{r['f1_score']:.2f}",
        'ROC-AUC (%)': f"{r['roc_auc']:.2f}"
    })

df_perf = pd.DataFrame(rows)
print('='*110)
print(f'{"TABLE: PERBANDINGAN PERFORMA — ABLATION STUDY":^110}')
print('='*110)
print(df_perf.to_string(index=False))
print('='*110)

In [ ]:
# TABLE 2: Efficiency
rows_eff = []
for r in ablation_results:
    rows_eff.append({
        'Model': r['model_name'],
        'Features': r['scenario'],
        'Training (s)': f"{r['train_time']:.2f}",
        'Inference/10k (s)': f"{r['inference_10k']:.4f}",
        'Model Size (MB)': f"{r['model_size_mb']:.2f}",
        'RAM (MB)': f"{r['ram_mb']:.1f}"
    })

df_eff = pd.DataFrame(rows_eff)
print('='*90)
print(f'{"TABLE: EFISIENSI — ABLATION STUDY":^90}')
print('='*90)
print(df_eff.to_string(index=False))
print('='*90)

## 4. Grafik: F1-Score vs Number of Features (per Model)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors = {'XGBoost': 'steelblue', 'Random Forest': 'forestgreen', 'SVM': 'coral'}
markers = {'XGBoost': 'o', 'Random Forest': 's', 'SVM': '^'}

for model_name in MODELS.keys():
    mr = [r for r in ablation_results if r['model_name'] == model_name]
    if not mr:
        continue
    n_feats = [r['n_features'] for r in mr]
    f1s = [r['f1_score'] for r in mr]
    
    ax.plot(n_feats, f1s, marker=markers[model_name], linestyle='-',
            color=colors[model_name], linewidth=2, markersize=8, label=model_name)
    for nf, f1 in zip(n_feats, f1s):
        ax.annotate(f'{f1:.1f}%', (nf, f1), textcoords='offset points',
                   xytext=(0, 10), ha='center', fontsize=8)

ax.set_xlabel('Number of Features')
ax.set_ylabel('F1-Score (%)')
ax.set_title('Ablation Study: F1-Score vs Number of Features\n(XGBoost vs Random Forest vs SVM)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.invert_xaxis()  # More features on left

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'ablation_f1_vs_features.png'), bbox_inches='tight')
plt.show()
print('Saved: ablation_f1_vs_features.png')

## 5. Grafik: Training Time vs Number of Features

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for model_name in MODELS.keys():
    mr = [r for r in ablation_results if r['model_name'] == model_name]
    if not mr:
        continue
    n_feats = [r['n_features'] for r in mr]
    times = [r['train_time'] for r in mr]
    
    ax.plot(n_feats, times, marker=markers[model_name], linestyle='--',
            color=colors[model_name], linewidth=2, markersize=8, label=model_name)
    for nf, t in zip(n_feats, times):
        ax.annotate(f'{t:.1f}s', (nf, t), textcoords='offset points',
                   xytext=(0, 10), ha='center', fontsize=8)

ax.set_xlabel('Number of Features')
ax.set_ylabel('Training Time (seconds)')
ax.set_title('Training Time vs Number of Features', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.invert_xaxis()

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'ablation_time_vs_features.png'), bbox_inches='tight')
plt.show()
print('Saved: ablation_time_vs_features.png')

## 6. Confusion Matrix — Best Configuration

In [ ]:
# Best overall F1
best = max(ablation_results, key=lambda x: x['f1_score'])
print(f'Best config: {best["model_name"]} — {best["scenario"]} → F1={best["f1_score"]:.2f}%')

cm = confusion_matrix(best['y_test'], best['y_pred'])
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
class_names = list(label_mapping.keys())

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
            xticklabels=class_names, yticklabels=class_names)
ax.set_title(f'Confusion Matrix (Normalized)\n{best["model_name"]} — {best["scenario"]}', fontweight='bold', fontsize=12)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'confusion_matrix_best_ablation.png'), bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix_best_ablation.png')

## 7. Feature Importance — Top-10 (untuk deployment)

In [ ]:
feat_imp_df = prev_results['feature_importance']
top10_df = feat_imp_df.head(10).iloc[::-1]

fig, ax = plt.subplots(figsize=(9, 5))
colors_bar = plt.cm.viridis(np.linspace(0.3, 0.9, 10))
ax.barh(range(10), top10_df['importance'], color=colors_bar, edgecolor='black', linewidth=0.5)
ax.set_yticks(range(10))
ax.set_yticklabels(top10_df['feature'], fontsize=10)
ax.set_xlabel('Feature Importance (Gain)')
ax.set_title('Top-10 Features for Deployment\nCSE-CIC-IDS2018', fontsize=13, fontweight='bold')

for i, val in enumerate(top10_df['importance']):
    ax.text(val + 0.001, i, f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'feature_importance_top10_deploy.png'), bbox_inches='tight')
plt.show()
print('Saved: feature_importance_top10_deploy.png')

## 8. Narasi & Kesimpulan Ablation Study

In [ ]:
print('='*70)
print(f'{"NARASI ABLATION STUDY":^70}')
print('='*70)

# Per-model, per-scenario analysis
for model_name in MODELS.keys():
    mr = [r for r in ablation_results if r['model_name'] == model_name]
    if not mr:
        continue
    print(f'\n■ {model_name}:')
    for r in mr:
        print(f"  {r['scenario']:15s} ({r['n_features']:>2d} feat) → F1={r['f1_score']:.2f}% | Time={r['train_time']:.2f}s | Size={r['model_size_mb']:.1f}MB")
    
    best_m = max(mr, key=lambda x: x['f1_score'])
    worst_m = min(mr, key=lambda x: x['f1_score'])
    print(f"  → Drop dari All ke Top-10: {best_m['f1_score'] - worst_m['f1_score']:.2f}% F1")

# Overall
best_all = max(ablation_results, key=lambda x: x['f1_score'])
best_efficient = max(
    [r for r in ablation_results if r['scenario'] == 'Top-10'],
    key=lambda x: x['f1_score']
) if [r for r in ablation_results if r['scenario'] == 'Top-10'] else None

print(f"""
{'='*70}
■ KESIMPULAN ABLATION STUDY:

  1. BEST OVERALL: {best_all['model_name']} ({best_all['scenario']}) → F1={best_all['f1_score']:.2f}%
  
  2. BEST EFFICIENT (Top-10): {best_efficient['model_name'] if best_efficient else 'N/A'} → F1={best_efficient['f1_score']:.2f}% (hanya 10 fitur)
  
  3. TRADE-OFF:
     - Pengurangan dari All→Top-20: minimal loss (<1% F1 typical)
     - Pengurangan dari All→Top-10: moderate loss tapi masih acceptable
     - Training time berkurang signifikan dengan fewer features
     - Model size berkurang → lebih cepat untuk deployment
  
  4. REKOMENDASI DEPLOYMENT:
     - Jika prioritas akurasi: gunakan All Features + XGBoost
     - Jika prioritas speed/size: gunakan Top-10 + XGBoost
     - Sweet spot: Top-15 (balance antara performa & efisiensi)

{'='*70}
""")

In [ ]:
# Save ablation results
ablation_export = {
    'ablation_results': [{k: v for k, v in r.items() if k not in ['y_test', 'y_pred']} for r in ablation_results],
    'top20_features': top20_features,
    'top15_features': top15_features,
    'top10_features': top10_features,
    'best_config': {'model': best_all['model_name'], 'scenario': best_all['scenario'], 'f1': best_all['f1_score']}
}

with open(os.path.join(DATA_DIR, 'ablation_results_04.pkl'), 'wb') as f:
    pickle.dump(ablation_export, f)

# Save tables as CSV
df_perf.to_csv(os.path.join(DATA_DIR, 'ablation_performance.csv'), index=False)
df_eff.to_csv(os.path.join(DATA_DIR, 'ablation_efficiency.csv'), index=False)

print('Saved:')
print('  ablation_results_04.pkl')
print('  ablation_performance.csv')
print('  ablation_efficiency.csv')

## 9. Export Model Terbaik untuk AWS Deployment

Simpan **hanya model XGBoost terbaik** per skenario fitur (10, 15, 20).

Format:
- Model: XGBoost native `.json` (portable, tanpa pickle, cross-platform)
- Metadata: `.json` berisi scaler params, feature list, label mapping

**Total: 6 file** (3 model + 3 metadata) → siap upload ke S3 untuk Lambda/EC2.

In [ ]:
import json

DEPLOY_DIR = os.path.join(MODEL_DIR, 'deploy')
os.makedirs(DEPLOY_DIR, exist_ok=True)

print('='*70)
print(f'{"EXPORT TOP-3 MODEL TERBAIK UNTUK AWS":^70}')
print('='*70)

# Rank semua ablation results by F1-Score (descending)
ranked = sorted(ablation_results, key=lambda x: x['f1_score'], reverse=True)

print('\nRanking semua eksperimen (by F1):')
for i, r in enumerate(ranked):
    marker = ' ← DEPLOY' if i < 3 else ''
    print(f"  #{i+1:>2d}  {r['model_name']:15s} | {r['scenario']:15s} ({r['n_features']:>2d} feat) | F1={r['f1_score']:.2f}%{marker}")

# Top-3
top3 = ranked[:3]
print(f'\n→ Deploying top-3 models to {DEPLOY_DIR}')

In [ ]:
print(f'\n{"="*70}')
print(f'{"RE-TRAINING & EXPORTING TOP-3":^70}')
print(f'{"="*70}')

# Map scenario name → feature indices
scenario_to_idx = {
    f'All ({len(all_feature_names)})': None,
    'Top-20': idx_top20,
    'Top-15': idx_top15,
    'Top-10': idx_top10
}
scenario_to_feats = {
    f'All ({len(all_feature_names)})': all_feature_names,
    'Top-20': top20_features,
    'Top-15': top15_features,
    'Top-10': top10_features
}

deploy_summary = []

for rank_idx, r in enumerate(top3):
    model_name = r['model_name']
    scenario = r['scenario']
    feat_idx = scenario_to_idx[scenario]
    feat_list = scenario_to_feats[scenario]
    
    X_scenario = X_all[:, feat_idx] if feat_idx is not None else X_all
    X_train, X_test, y_train, y_test = train_test_split(
        X_scenario, y_all, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y_all)
    
    # Re-train the exact model
    model = MODELS[model_name]()
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start
    
    y_pred = model.predict(X_test)
    f1_val = f1_score(y_test, y_pred, average='weighted', zero_division=0) * 100
    acc_val = accuracy_score(y_test, y_pred) * 100
    
    # Inference benchmark
    n_bench = min(10000, len(X_test))
    start = time.time()
    _ = model.predict(X_test[:n_bench])
    inf_10k = (time.time() - start) / n_bench * 10000
    
    # File naming
    tag = f"rank{rank_idx+1}_{model_name.lower().replace(' ','_')}_{scenario.lower().replace(' ','_').replace('(','').replace(')','')}"
    
    # Save model
    if model_name == 'XGBoost':
        # Native JSON (preferred for XGBoost)
        model_filename = f'{tag}.json'
        model_path = os.path.join(DEPLOY_DIR, model_filename)
        model.save_model(model_path)
    else:
        # RF/SVM → pickle (no native json support)
        model_filename = f'{tag}.pkl'
        model_path = os.path.join(DEPLOY_DIR, model_filename)
        with open(model_path, 'wb') as f:
            pickle.dump(model, f)
    
    model_size = os.path.getsize(model_path) / (1024*1024)
    
    # Save metadata
    feat_idx_list = feat_idx if feat_idx is not None else list(range(len(all_feature_names)))
    metadata = {
        'rank': rank_idx + 1,
        'model_type': model_name,
        'model_file': model_filename,
        'dataset': 'CSE-CIC-IDS2018',
        'feature_scenario': scenario,
        'feature_names': feat_list if isinstance(feat_list, list) else list(feat_list),
        'n_features': len(feat_list),
        'n_classes': n_classes,
        'label_mapping': label_mapping,
        'inverse_label_mapping': {str(v): k for k, v in label_mapping.items()},
        'scaler': {
            'type': 'StandardScaler',
            'mean': data['scaler'].mean_[feat_idx_list].tolist(),
            'scale': data['scaler'].scale_[feat_idx_list].tolist()
        },
        'performance': {
            'f1_score_pct': round(f1_val, 2),
            'accuracy_pct': round(acc_val, 2),
            'train_time_sec': round(train_time, 2),
            'inference_10k_sec': round(inf_10k, 4),
            'model_size_mb': round(model_size, 2)
        }
    }
    
    meta_filename = f'{tag}_meta.json'
    meta_path = os.path.join(DEPLOY_DIR, meta_filename)
    with open(meta_path, 'w') as mf:
        json.dump(metadata, mf, indent=2)
    
    print(f'\n  ■ RANK #{rank_idx+1}: {model_name} — {scenario} ({len(feat_list)} features)')
    print(f'    F1={f1_val:.2f}% | Acc={acc_val:.2f}% | Inf/10k={inf_10k:.4f}s | Size={model_size:.2f}MB')
    print(f'    Model: {model_filename}')
    print(f'    Meta:  {meta_filename}')
    
    deploy_summary.append({
        'rank': rank_idx+1, 'model': model_name, 'scenario': scenario,
        'n_features': len(feat_list), 'f1': f1_val, 'accuracy': acc_val,
        'inference_10k': inf_10k, 'model_file': model_filename,
        'meta_file': meta_filename, 'model_size_mb': model_size
    })

print(f'\n{"="*70}')
print(f'DEPLOYMENT SUMMARY — {DEPLOY_DIR}')
print(f'{"="*70}')
print(f'\n{"#":>3s} {"Model":15s} {"Scenario":12s} {"Feat":>5s} {"F1":>8s} {"Inf/10k":>10s} {"Size":>8s}')
print(f'{"-"*70}')
for d in deploy_summary:
    print(f"  {d['rank']:>1d}  {d['model']:15s} {d['scenario']:12s} {d['n_features']:>4d}  {d['f1']:.2f}%  {d['inference_10k']:.4f}s  {d['model_size_mb']:.2f}MB")

print(f'\nTotal files: {len(deploy_summary) * 2} (3 models + 3 metadata)')
print(f'\nUntuk uji coba di AWS:')
print(f'  1. Upload folder models/deploy/ ke S3')
print(f'  2. Bisa test ketiga model secara parallel di Lambda/EC2')
print(f'  3. Bandingkan real-world inference speed & accuracy')
print(f'  4. Pilih final model berdasarkan hasil AWS testing')